# DACON 구내식당 식수 예측

날짜와 근무 인원 정보를 기반으로 중식계와 석식계를 예측하는 모델링 흐름을 요약합니다.

## 1. 데이터 로드

대회 원본 CSV 파일을 `data/raw` 아래에 배치한 뒤 실행합니다.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from siksu.config import TRAIN_FILE, TEST_FILE, SAMPLE_SUBMISSION_FILE
from siksu.pipeline import load_competition_data

train, test, sample_submission = load_competition_data(
    train_path=TRAIN_FILE,
    test_path=TEST_FILE,
    sample_submission_path=SAMPLE_SUBMISSION_FILE,
)

train.head()

## 2. 피처 엔지니어링

- `일자`에서 월과 일을 추출합니다.
- 한국어 요일 값을 숫자형으로 인코딩합니다.
- 정원에서 휴가, 출장, 재택근무 인원을 제외해 `현재원`을 생성합니다.

In [ ]:
from siksu.features import prepare_train_test, build_feature_matrix

train_features, test_features = prepare_train_test(train, test)
feature_matrix = build_feature_matrix(train_features)

feature_matrix.head()

## 3. 학습 및 제출 파일 생성

중식과 석식은 별도 XGBoost 회귀 모델로 학습합니다. 제출 파일 생성은 CLI와 동일한 파이프라인을 호출합니다.

In [ ]:
from siksu.pipeline import run_pipeline

result = run_pipeline(use_grid_search=True)
result